# MMLU-style log-probability scoring with GPT-2

Companion notebook for [What is MMLU, and how does log-probability scoring work?](https://gangwar-ajay.github.io/Interview-Preparation/questions/mmlu-logprob.html)

We score each candidate answer by the log-probability the model assigns to it as a continuation of the prompt — no text generation at all — then compare raw-sum vs length-normalized picks.

In [ ]:
%pip -q install transformers torch

In [ ]:
import torch, torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()

def continuation_logprob(prompt, answer):
    """Sum of log p(token) over the answer tokens, given the prompt."""
    p_ids = tok(prompt, return_tensors="pt").input_ids
    a_ids = tok(answer, return_tensors="pt").input_ids
    ids = torch.cat([p_ids, a_ids], dim=1)
    with torch.no_grad():
        logits = model(ids).logits
    logprobs = F.log_softmax(logits[0, :-1], dim=-1)   # position i predicts token i+1
    answer_positions = range(p_ids.shape[1] - 1, ids.shape[1] - 1)
    lp = [logprobs[pos, ids[0, pos + 1]].item() for pos in answer_positions]
    return sum(lp), len(lp)

In [ ]:
prompt = ("Question: What is the time complexity of binary search "
          "on a sorted array?\nAnswer: ")

results = {}
for ans in ["O(n)", "O(log n)", "O(n log n)", "O(1)"]:
    raw, n = continuation_logprob(prompt, ans)
    results[ans] = (raw, raw / n, n)
    print(f"{ans:12s} raw {raw:8.3f}   per-token {raw / n:7.3f}   ({n} tokens)")

print("\nraw-sum pick:        ", max(results, key=lambda a: results[a][0]))
print("length-normalized pick:", max(results, key=lambda a: results[a][1]))

## Letter scoring — the official MMLU method

Compare the next-token probabilities of " A", " B", " C", " D" after a prompt that lists the options. Note the leading space: `" B"` and `"B"` are different GPT-2 tokens — forgetting this is the classic harness bug.

In [ ]:
letter_prompt = ("Question: What is the time complexity of binary search on a sorted array?\n"
                 "A. O(n)\nB. O(log n)\nC. O(n log n)\nD. O(1)\n"
                 "Answer:")

ids = tok(letter_prompt, return_tensors="pt").input_ids
with torch.no_grad():
    next_logprobs = F.log_softmax(model(ids).logits[0, -1], dim=-1)

for letter in [" A", " B", " C", " D"]:
    t = tok(letter).input_ids[0]
    print(f"'{letter}' (token {t}): {next_logprobs[t].item():7.3f}")